# UK Used Car Price Prediction: 03. Preprocessing dan Feature Engineering

Notebook ini membagi data bersih menjadi train/test, membuat pipeline preprocessing yang dapat diulang, dan melatih baseline. Semua preprocessing di-fit hanya pada data training untuk mencegah data leakage.

**Input:** `data/processed/cars_clean.csv` dari notebook 01.  \n
**Output:** hasil baseline dan `models/03_ridge_baseline.joblib`.

Notebook berikutnya (`04_modeling_evaluation.ipynb`) memakai split dan metrik yang sama untuk membandingkan model lanjutan.

## 1. Setup dan Load Data

In [1]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.dummy import DummyRegressor

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluate import score_regression
from src.provenance import save_model
from src.train import DATA_PATH, MODEL_PATH, build_model, load_evaluation_split

## 2. Train/Test Split

Kolom `price` adalah target. Test set disisihkan sebelum preprocessing dan tidak dipakai saat memilih transformasi maupun parameter baseline.

In [2]:
(X_train, X_test, y_train, y_test), evaluation_input = load_evaluation_split(DATA_PATH)
row_count = evaluation_input["dataset_rows"]
split_summary = pd.DataFrame(
    {"baris": [len(X_train), len(X_test)],
     "proporsi": [len(X_train) / row_count, len(X_test) / row_count],
     "median_price": [y_train.median(), y_test.median()]},
    index=["train", "test"],
)
display(split_summary.style.format({"proporsi": "{:.1%}", "median_price": "£{:,.0f}"}))
print("SHA-256 CSV:", evaluation_input["dataset_sha256"])

,baris,proporsi,median_price
train,77928,80.0%,"£14,480"
test,19483,20.0%,"£14,400"


SHA-256 CSV: fa8ba78acc33dff414b6dc612fed54bbd67fc7fbf41b763a55fcac6df9c26c8f


## 3. Pipeline Preprocessing

`year` diubah menjadi `car_age` agar maknanya lebih langsung untuk model, kemudian kolom `year` asli dihapus. Fitur numerik diskalakan untuk Ridge. Fitur kategorikal di-one-hot encode; kategori model dengan frekuensi kurang dari 10 digabung oleh `OneHotEncoder(min_frequency=10)`. Kategori yang belum pernah muncul saat training diabaikan saat prediksi, bukan menyebabkan error.

In [3]:
ridge_log = build_model().fit(X_train, y_train)
feature_pipeline = ridge_log.regressor_.named_steps["features"]
print(f"Matriks training: {feature_pipeline.transform(X_train).shape}")
print(f"Matriks test    : {feature_pipeline.transform(X_test).shape}")

Matriks training: (77928, 178)
Matriks test    : (19483, 178)


## 4. Baseline

Dua pembanding dipakai:

1. **Median dummy:** selalu memprediksi median harga training.
2. **Ridge + log target:** baseline linear yang memakai semua fitur hasil pipeline. `TransformedTargetRegressor` menerapkan `log1p(price)` saat fit dan mengembalikan prediksi ke satuan pound saat prediksi.

Nilai `alpha=10` adalah baseline tetap, bukan hasil tuning; tuning dilakukan pada tahap modelling.

In [4]:
dummy = DummyRegressor(strategy="median").fit(X_train, y_train)
predictions = {
    "Median dummy": dummy.predict(X_test),
    "Ridge + log(price)": ridge_log.predict(X_test),
}
results = pd.DataFrame(
    {name: score_regression(y_test, pred) for name, pred in predictions.items()}
).T.sort_values("MAE (£)")
display(results.style.format({"MAE (£)": "£{:,.0f}", "RMSE (£)": "£{:,.0f}", "R²": "{:.3f}"}))

,MAE (£),RMSE (£),R²
Ridge + log(price),"£1,651","£3,090",0.905
Median dummy,"£6,854","£10,271",-0.051


## 5. Simpan Baseline dan Catatan untuk Tahap Berikutnya

In [5]:
provenance = save_model(ridge_log, MODEL_PATH, evaluation_input)
print(f"Baseline tersimpan: {MODEL_PATH.relative_to(PROJECT_ROOT)}")
print("SHA-256 model:", provenance["sha256"])

Baseline tersimpan: models\03_ridge_baseline.joblib
SHA-256 model: f09b0ccf38c2692d055ca8b27bdd38d20ac655fc0a8d575d60fe7eb7e9beca48
